In [1]:
import json, time, warnings, os, sys
from dataclasses import dataclass
from typing import Tuple, List
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, RationalQuadratic, WhiteKernel
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# ================================================================
# 🧠 Setup
# ================================================================
sys.setrecursionlimit(2000)
warnings.filterwarnings("ignore", category=UserWarning)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# ================================================================
# 1️⃣ Data Loader
# ================================================================
def load_data_for_function(function_number):
    base_path = "initial_data"
    inputs_path = os.path.join(base_path, f'function_{function_number}', 'initial_inputs.npy')
    outputs_path = os.path.join(base_path, f'function_{function_number}', 'initial_outputs.npy')
    X0 = np.load(inputs_path)
    y0 = np.load(outputs_path)

    weekly_base_path = "weekly_response\\week3"
    loaded_inputs = np.load(os.path.join(weekly_base_path, "inputs.npy"), allow_pickle=True)
    loaded_outputs = np.load(os.path.join(weekly_base_path, "outputs.npy"), allow_pickle=True)
    for week_inputs, week_outputs in zip(loaded_inputs, loaded_outputs):
        X0 = np.vstack([X0, week_inputs[function_number - 1]])
        y0 = np.hstack([y0, week_outputs[function_number - 1]])

    scale_factor = 1.0
    if np.max(np.abs(y0)) < 1e-6 or np.var(y0) < 1e-10:
        scale_factor = 1e6
        y0 = y0 * scale_factor
        print(f"[F{function_number}] ⚙️ Auto-rescaled outputs ×{scale_factor:g}")
    return X0, y0, scale_factor

# ================================================================
# 2️⃣ Acquisition Functions
# ================================================================
def ei(mu, sigma, y_best, beta=1.5):
    sigma = np.maximum(sigma, 1e-12)
    z = (mu - y_best) / (beta * sigma)
    return (mu - y_best) * norm.cdf(z) + (beta * sigma) * norm.pdf(z)

def ucb(mu, sigma, kappa=2.5):
    return mu + kappa * sigma

def hybrid_acquisition(mu, sigma, y_best, beta=1.5, w_ei=0.7, w_ucb=0.3, kappa=2.5):
    ei_val = ei(mu, sigma, y_best, beta=beta)
    ucb_val = ucb(mu, sigma, kappa=kappa)
    ei_val = (ei_val - np.min(ei_val)) / (np.ptp(ei_val) + 1e-12)
    ucb_val = (ucb_val - np.min(ucb_val)) / (np.ptp(ucb_val) + 1e-12)
    return w_ei * ei_val + w_ucb * ucb_val

# ================================================================
# 3️⃣ Bayesian Neural Network Surrogate
# ================================================================
class BNNRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)

class BayesianNN:
    def __init__(self, input_dim, epochs=300, lr=1e-3, base_dropout=0.1, base_mc=30, out_dir=None, prefix=""):
        if input_dim <= 4:
            dropout, mc_samples = base_dropout, base_mc
        elif input_dim <= 6:
            dropout, mc_samples = base_dropout + 0.05, base_mc + 10
        else:
            dropout, mc_samples = base_dropout + 0.1, base_mc + 20
        self.input_dim = input_dim
        self.mc_samples = mc_samples
        self.model = BNNRegressor(input_dim, hidden_dim=64 + 8 * max(0, input_dim - 3), dropout=dropout)
        self.optim = torch.optim.Adam(self.model.parameters(), lr=lr)
        self.epochs = epochs
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        print(f"{prefix} [BNN-D{input_dim}] Dropout={dropout:.2f}, MC={mc_samples}, Epochs={epochs}")

    def fit(self, X, y):
        X = torch.tensor(X, dtype=torch.float32, device=self.device)
        y = torch.tensor(y, dtype=torch.float32, device=self.device).view(-1, 1)
        self.model.train()
        for _ in range(self.epochs):
            self.optim.zero_grad()
            loss = F.mse_loss(self.model(X), y)
            loss.backward()
            self.optim.step()

    def predict(self, X):
        X = torch.tensor(X, dtype=torch.float32, device=self.device)
        self.model.train()
        preds = []
        with torch.no_grad():
            for _ in range(self.mc_samples):
                preds.append(self.model(X).cpu().numpy())
        preds = np.stack(preds, axis=0)
        return preds.mean(axis=0).flatten(), preds.std(axis=0).flatten()

# ================================================================
# 4️⃣ Agent Config + Exploration Controller
# ================================================================
@dataclass
class AgentConfig:
    bounds: List[Tuple[float, float]]
    n_candidates: int = 4000
    edge_epsilon: float = 1e-2

class ExplorationController:
    def __init__(self, fn):
        self.fn = fn
        self.low_pi_count = 0
        self.mode = "normal"
        self.pi_history = []

    def update(self, last_pi):
        self.pi_history.append(last_pi)
        if last_pi < 0.3:
            self.low_pi_count += 1
        else:
            self.low_pi_count = 0
        if self.low_pi_count >= 2 and self.mode != "high":
            self.mode = "high"
            print(f"[F{self.fn}] 🔄 Switching to HIGH exploration.")
        elif last_pi > 0.5 and self.mode == "high":
            self.mode = "normal"
            print(f"[F{self.fn}] 🟢 Returning to NORMAL exploration.")

    def get_params(self):
        return {"beta": 3.0, "w_ei": 0.5, "w_ucb": 0.5, "sigma_boost": 1.8} if self.mode == "high" else \
               {"beta": 1.5, "w_ei": 0.7, "w_ucb": 0.3, "sigma_boost": 1.0}

# ================================================================
# 5️⃣ Hybrid Agentic BO
# ================================================================
class HybridAgenticBO:
    def __init__(self, X, y, cfg, out_dir, fn, D, scale_factor):
        self.X, self.y, self.cfg, self.out_dir = X, y, cfg, out_dir
        os.makedirs(out_dir, exist_ok=True)
        self.fn, self.D = fn, D
        self.scale_factor = scale_factor
        self.prefix = f"[F{fn}-D{D}]"
        self.scaler_x = StandardScaler().fit(X)
        self.scaler_y = StandardScaler().fit(y.reshape(-1, 1))
        self.Xs = self.scaler_x.transform(X)
        self.ys = self.scaler_y.transform(y.reshape(-1, 1)).ravel()
        self.model_name, self.model, self.acq = None, None, None
        self.explore_ctrl = ExplorationController(fn)

    def select_model(self):
        if self.D >= 4:
            self.model_name = "BNN"
            self.model = BayesianNN(self.D, epochs=200, base_dropout=0.1, base_mc=25,
                                    out_dir=self.out_dir, prefix=self.prefix)
            print(f"{self.prefix} 🧠 Using Bayesian Neural Network")
            return
        kernels = [Matern(nu=2.5) + WhiteKernel(1e-10), RBF(length_scale=0.3) + WhiteKernel(1e-10),
                   RationalQuadratic(alpha=1.0, length_scale=0.3) + WhiteKernel(1e-10)]
        best_rmse, best_gp = 1e9, None
        kf = KFold(n_splits=min(3, len(self.Xs)), shuffle=True, random_state=42)
        for k in kernels:
            gp = GaussianProcessRegressor(kernel=k, normalize_y=True, n_restarts_optimizer=1)
            rmse = 0
            for tr, vl in kf.split(self.Xs):
                gp.fit(self.Xs[tr], self.ys[tr])
                preds = gp.predict(self.Xs[vl])
                rmse += np.mean((preds - self.ys[vl]) ** 2)
            rmse = np.sqrt(rmse / 3)
            if rmse < best_rmse:
                best_rmse, best_gp = rmse, gp
        self.model_name, self.model = "GP", best_gp
        print(f"{self.prefix} 🧠 Using Gaussian Process (best RMSE kernel)")

    def fit(self):
        print(f"{self.prefix} 🔧 Fitting {self.model_name} ...")
        self.model.fit(self.Xs, self.ys)
        print(f"{self.prefix} ✅ Model training complete.")

    def predict_mu_sigma(self, Xq):
        Xqs = self.scaler_x.transform(Xq)
        mu, sig = (self.model.predict(Xqs, return_std=True) if self.model_name == "GP" else self.model.predict(Xqs))
        mu = self.scaler_y.inverse_transform(mu.reshape(-1, 1)).ravel() / self.scale_factor
        sig = sig * float(self.scaler_y.scale_[0]) / self.scale_factor
        return mu, sig

    def choose_acquisition(self, mu, sigma):
        sigma_ratio = float(np.std(sigma) / max(np.std(mu), 1e-9))
        self.acq = "UCB" if (np.std(sigma) > 0.4 * np.std(mu) or sigma_ratio < 0.05) else "EI"
        print(f"{self.prefix} 🎯 Acquisition: {self.acq} (σ_ratio={sigma_ratio:.3f})")
        return sigma_ratio

    def propose(self):
        eps = self.cfg.edge_epsilon
        lows = np.array([b[0] + eps for b in self.cfg.bounds])
        highs = np.array([b[1] - eps for b in self.cfg.bounds])
        params = self.explore_ctrl.get_params()
        beta, w_ei, w_ucb, sigma_boost = params["beta"], params["w_ei"], params["w_ucb"], params["sigma_boost"]
        cand = np.random.uniform(lows, highs, size=(self.cfg.n_candidates, self.D))
        mu, sig = self.predict_mu_sigma(cand)
        sig = sig * sigma_boost
        y_best = np.max(self.y) / self.scale_factor
        acq_vals = hybrid_acquisition(mu, sig, y_best, beta=beta, w_ei=w_ei, w_ucb=w_ucb) if self.acq == "EI" else ucb(mu, sig)
        best_x = cand[np.argmax(acq_vals)]
        mu_b, sig_b = self.predict_mu_sigma(best_x.reshape(1, -1))
        sig_b = sig_b * sigma_boost
        ei_b = ei(np.array([mu_b[0]]), np.array([sig_b[0]]), y_best, beta=beta)[0]
        pi_b = norm.cdf((mu_b[0] - y_best) / max(sig_b[0], 1e-12))
        print(f"{self.prefix} 💡 Proposed x={np.round(best_x,6)} μ={mu_b[0]:.3e}, σ={sig_b[0]:.3e}, EI={ei_b:.3e}, PI={pi_b:.3f} (Mode={self.explore_ctrl.mode})")
        self.explore_ctrl.update(pi_b)
        return best_x, float(mu_b[0]), float(sig_b[0]), float(pi_b), float(ei_b)

# ================================================================
# 6️⃣ JSON-safe Helper
# ================================================================
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    elif isinstance(obj, (np.generic,)):
        return obj.item()
    elif hasattr(obj, "tolist"):
        return obj.tolist()
    elif isinstance(obj, (float, int, str, bool)) or obj is None:
        return obj
    else:
        try:
            return float(obj)
        except Exception:
            return str(obj)

# ================================================================
# 7️⃣ Multi-Round Autonomous Optimization + Convergence Plots
# ================================================================
def run_all_functions():
    dim_map = {1:2, 2:2, 3:3, 4:4, 5:4, 6:5, 7:6, 8:8}
    results_root = "results_wk4_agentic_autonomous_plots"
    os.makedirs(results_root, exist_ok=True)
    all_results = []
    MAX_ROUNDS = 8
    STOP_THRESHOLD = 0.12

    for fn in range(1, 9):
        D = dim_map[fn]
        prefix = f"[F{fn}-D{D}]"
        print(f"\n{'='*100}\n{prefix} 🚀 Starting Function {fn}\n{'='*100}")
        start = time.time()
        X0, y0, scale_factor = load_data_for_function(fn)
        prev_idx = int(np.argmax(y0))
        prev_x, prev_y = X0[prev_idx], float(y0[prev_idx] / scale_factor)
        print(f"{prefix} 📈 Previous best input: {np.round(prev_x,6)}")
        print(f"{prefix} 📈 Previous best output: {prev_y:.6e}")
        cfg = AgentConfig(bounds=[(0,1)] * D, n_candidates=4000 if D < 4 else 2000)
        agent = HybridAgenticBO(X0, y0, cfg, os.path.join(results_root, f"function_{fn}"), fn, D, scale_factor)
        agent.select_model(); agent.fit()

        round_records, pi_list, ei_list, modes = [], [], [], []
        for r in range(1, MAX_ROUNDS + 1):
            print(f"\n{prefix} 🔁 Round {r}/{MAX_ROUNDS}")
            mu_all, sigma_all = agent.predict_mu_sigma(agent.X)
            sigma_ratio = agent.choose_acquisition(mu_all, sigma_all)
            prop_x, prop_mu, prop_sig, prop_pi, prop_ei = agent.propose()
            reasoning = f"{prefix} Round {r}. Model={agent.model_name}, Acq={agent.acq}, Mode={agent.explore_ctrl.mode}, μ={prop_mu:.3e}, σ={prop_sig:.3e}, EI={prop_ei:.3e}, PI={prop_pi:.3f}"
            print(f"{prefix} 🧠 Reasoning: {reasoning}")
            pi_list.append(prop_pi); ei_list.append(prop_ei); modes.append(agent.explore_ctrl.mode)
            if prop_pi < STOP_THRESHOLD:
                print(f"{prefix} 🛑 Converged early (PI={prop_pi:.3f} < {STOP_THRESHOLD})"); break
            y_new = prop_mu + np.random.normal(0, prop_sig * 0.5)
            X_new = np.array(prop_x).reshape(1, -1)
            agent.X = np.vstack([agent.X, X_new]); agent.y = np.hstack([agent.y, y_new])
            agent.Xs = agent.scaler_x.fit_transform(agent.X)
            agent.ys = agent.scaler_y.fit_transform(agent.y.reshape(-1, 1)).ravel(); agent.fit()
            round_records.append({"round": r, "proposed_x": prop_x.tolist(), "predicted_mu": prop_mu, "predicted_sigma": prop_sig, "predicted_pi": prop_pi, "predicted_ei": prop_ei, "exploration_mode": agent.explore_ctrl.mode, "reasoning": reasoning})

        out_dir = os.path.join(results_root, f"function_{fn}")
        plt.figure(figsize=(7,5))
        plt.plot(range(1, len(pi_list)+1), pi_list, "-o", label="PI")
        plt.plot(range(1, len(ei_list)+1), ei_list, "-o", label="EI")
        for i,m in enumerate(modes): plt.text(i+1, pi_list[i], m[0].upper(), fontsize=8, color="gray")
        plt.title(f"Convergence - Function {fn} ({D}D)"); plt.xlabel("Round"); plt.ylabel("Improvement")
        plt.legend(); plt.grid(True, ls="--", alpha=0.6); plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"convergence_F{fn}.png"), dpi=300); plt.close()
        print(f"{prefix} 📊 Convergence plot saved → {out_dir}/convergence_F{fn}.png")

        elapsed = time.time()-start; avg_pi = np.mean(agent.explore_ctrl.pi_history) if agent.explore_ctrl.pi_history else 0
        summary={"function":fn,"dimension":D,"rounds_run":len(round_records),"previous_best_input":prev_x.tolist(),"previous_best_output":prev_y,"model_used":agent.model_name,"final_exploration_mode":agent.explore_ctrl.mode,"average_pi_across_rounds":round(float(avg_pi),3),"time_taken_seconds":round(elapsed,2),"round_records":round_records}
        with open(os.path.join(out_dir,"multi_round_summary.json"),"w") as f: json.dump(make_json_safe(summary),f,indent=2)
        all_results.append(make_json_safe(summary))
        print(f"{prefix} ✅ Complete ⏱️ {elapsed:.2f}s | Avg PI={avg_pi:.3f}")

    df=pd.DataFrame(all_results); csv_path=os.path.join(results_root,"summary_results.csv")
    df.to_csv(csv_path,index=False)
    print("\n📊 Autonomous Multi-Round Summary saved to:",os.path.abspath(csv_path))

# ================================================================
# 9️⃣ LLM-style Reasoning Summarizer
# ================================================================
def generate_agentic_summaries(results_root="results_wk4_agentic_autonomous_plots"):
    print("\n🧠 Generating LLM-style reasoning summaries...\n")
    summary_files=[os.path.join(r,f) for r,_,fs in os.walk(results_root) for f in fs if f=="multi_round_summary.json"]
    global_summaries=[]
    for path in sorted(summary_files):
        data=json.load(open(path)); fn=data["function"]; D=data["dimension"]; rounds=data["rounds_run"]
        model=data["model_used"]; mode=data["final_exploration_mode"]; avg_pi=data["average_pi_across_rounds"]; recs=data["round_records"]
        if not recs: continue
        first,last=recs[0],recs[-1]
        ei_trend="increasing" if last["predicted_ei"]>first["predicted_ei"] else "decreasing"
        pi_trend="increasing" if last["predicted_pi"]>first["predicted_pi"] else "decreasing"
        behavior="stable convergence" if avg_pi<0.2 else "moderate confidence" if avg_pi<0.5 else "high confidence in local maxima"
        narrative=(f"**Summary (Function {fn}, {D}D)**\n"
                   f"The optimizer employed a {model} surrogate in {mode} mode for {rounds} rounds.\n"
                   f"Across iterations, Expected Improvement was {ei_trend}, while Probability of Improvement was {pi_trend}.\n"
                   f"Final average PI={avg_pi:.2f}, indicating {behavior}.\n"
                   f"Early rounds: EI≈{first['predicted_ei']:.3e}, PI≈{first['predicted_pi']:.2f}; "
                   f"Later rounds: EI≈{last['predicted_ei']:.3e}, PI≈{last['predicted_pi']:.2f}.\n"
                   f"The agent showed adaptive reasoning and self-correcting dynamics.")
        txt_path=os.path.join(os.path.dirname(path),"agentic_summary.txt")
        open(txt_path,"w",encoding="utf-8").write(narrative)
        print(f"[F{fn}-D{D}] 🧾 Summary saved → {txt_path}")
        global_summaries.append({"function":fn,"dimension":D,"avg_pi":avg_pi,"ei_trend":ei_trend,"pi_trend":pi_trend,"behavior":behavior,"summary_text":narrative})
    pd.DataFrame(global_summaries).to_csv(os.path.join(results_root,"agentic_reasoning_summary.csv"),index=False)
    print("\n📘 Global reasoning summary saved.")

# ================================================================
# 🔟 Diagnostics & Auto-Recommendations (Re-run Plans)
# ================================================================
def diagnose_runs(results_root="results_wk4_agentic_autonomous_plots"):
    """
    Scans each function's multi_round_summary.json, detects anomalies,
    and writes a recommended re-run plan that increases exploration
    when appropriate. Also saves a global diagnostics CSV.
    """
    print("\n🩺 Running diagnostics and generating re-run recommendations...\n")
    summary_files = [os.path.join(r, f) for r, _, fs in os.walk(results_root) for f in fs if f == "multi_round_summary.json"]

    diagnostics_rows = []
    global_plan = {}

    for path in sorted(summary_files):
        with open(path, "r") as f:
            data = json.load(f)

        fn = data["function"]
        D = data["dimension"]
        rounds_run = data.get("rounds_run", 0)
        model = data.get("model_used", "NA")
        mode_final = data.get("final_exploration_mode", "NA")
        avg_pi = float(data.get("average_pi_across_rounds", 0.0))
        recs = data.get("round_records", [])

        # Extract PI/EI time series
        pi_list = [float(r["predicted_pi"]) for r in recs] if recs else []
        ei_list = [float(r["predicted_ei"]) for r in recs] if recs else []
        modes = [r.get("exploration_mode", "normal") for r in recs] if recs else []

        # Metrics
        low_pi_ratio = (sum(p < 0.3 for p in pi_list) / len(pi_list)) if pi_list else 1.0
        high_mode_ratio = (sum(m == "high" for m in modes) / len(modes)) if modes else 0.0
        pi_trend = "flat"
        ei_trend = "flat"
        if len(pi_list) >= 2:
            pi_trend = "increasing" if (pi_list[-1] - pi_list[0]) > 0 else "decreasing" if (pi_list[-1] - pi_list[0]) < 0 else "flat"
        if len(ei_list) >= 2:
            ei_trend = "increasing" if (ei_list[-1] - ei_list[0]) > 0 else "decreasing" if (ei_list[-1] - ei_list[0]) < 0 else "flat"

        # Oscillation heuristic: many sign flips in first differences
        def count_sign_flips(xs):
            if len(xs) < 3: return 0
            diffs = np.diff(xs)
            signs = np.sign(diffs)
            flips = np.sum(signs[1:] * signs[:-1] < 0)
            return int(flips)
        pi_flips = count_sign_flips(pi_list)
        ei_flips = count_sign_flips(ei_list)
        oscillatory = (pi_flips + ei_flips) >= max(2, len(pi_list)//3)

        # Classify health
        health = "good"
        if avg_pi < 0.2 or low_pi_ratio > 0.6:
            health = "low_confidence"
        elif oscillatory or (ei_trend == "increasing" and pi_trend == "decreasing"):
            health = "unstable"

        # Build recommendation (defaults)
        plan = {
            "function": fn,
            "dimension": D,
            "model": model,
            "base_changes": {
                "n_candidates_factor": 1.0,
                "force_acq": None,              # e.g., "UCB"
                "beta": 1.5,
                "w_ei": 0.7,
                "w_ucb": 0.3,
                "sigma_boost": 1.0
            },
            "bnn_changes": None,               # only if BNN
            "gp_changes": None,                # only if GP
            "notes": []
        }

        # Recommendations by health
        if health == "low_confidence":
            plan["base_changes"].update({
                "n_candidates_factor": 1.5,
                "beta": 3.0,
                "w_ei": 0.5,
                "w_ucb": 0.5,
                "sigma_boost": 2.0
            })
            plan["notes"].append("Low average PI or many low-PI rounds → increase exploration, uncertainty, and candidate coverage.")
        elif health == "unstable":
            plan["base_changes"].update({
                "n_candidates_factor": 1.3,
                "beta": 2.2,
                "w_ei": 0.6,
                "w_ucb": 0.4,
                "sigma_boost": 1.5
            })
            plan["notes"].append("Oscillatory/contradictory trends → moderate exploration boost and more candidate smoothing.")
        else:
            plan["notes"].append("System behaved well; minor tuning only if runtime budget allows.")

        # Model-specific nits
        if model == "BNN":
            plan["bnn_changes"] = {
                "epochs_add": 50 if health != "good" else 0,
                "mc_samples_add": 10 if health != "good" else 0,
                "dropout_add": 0.05 if (health == "unstable" and D >= 6) else 0.0
            }
            if plan["bnn_changes"]["dropout_add"] > 0:
                plan["notes"].append("Increase dropout slightly to regularize high-D surrogate (stability).")
        elif model == "GP":
            plan["gp_changes"] = {
                "force_kernel_noise": True if health != "good" else False,
                "prefer_UCB_if_sigma_ratio_low": True
            }
            if plan["gp_changes"]["force_kernel_noise"]:
                plan["notes"].append("Add/strengthen WhiteKernel noise; prefer UCB if σ_ratio remains low.")

        # Persist per-function diagnostics
        diag = {
            "function": fn,
            "dimension": D,
            "rounds_run": rounds_run,
            "model": model,
            "final_mode": mode_final,
            "avg_pi": round(avg_pi, 3),
            "low_pi_ratio": round(low_pi_ratio, 3),
            "high_mode_ratio": round(high_mode_ratio, 3),
            "pi_trend": pi_trend,
            "ei_trend": ei_trend,
            "oscillatory": oscillatory,
            "health": health
        }
        diagnostics_rows.append(diag)

        fn_dir = os.path.dirname(path)
        with open(os.path.join(fn_dir, "diagnostics.json"), "w") as f:
            json.dump(make_json_safe(diag), f, indent=2)
        with open(os.path.join(fn_dir, "rerun_plan.json"), "w") as f:
            json.dump(make_json_safe(plan), f, indent=2)

        # Human-readable note
        txt = [
            f"Function {fn} ({D}D) — Health: {health}",
            f"Model: {model} | Final mode: {mode_final} | Rounds: {rounds_run}",
            f"Avg PI={avg_pi:.3f} | low_PI_ratio={low_pi_ratio:.2f} | high_mode_ratio={high_mode_ratio:.2f}",
            f"Trends: PI={pi_trend}, EI={ei_trend} | Oscillatory={oscillatory}",
            "Recommendation:",
            f"- Increase candidates x{plan['base_changes']['n_candidates_factor']:.1f}, beta={plan['base_changes']['beta']}, "
            f"w_ei={plan['base_changes']['w_ei']}, w_ucb={plan['base_changes']['w_ucb']}, sigma_boost={plan['base_changes']['sigma_boost']}",
        ]
        if plan["bnn_changes"]:
            bc = plan["bnn_changes"]
            txt.append(f"- BNN: +{bc['epochs_add']} epochs, +{bc['mc_samples_add']} MC, dropout+={bc['dropout_add']:.02f}")
        if plan["gp_changes"]:
            gc = plan["gp_changes"]
            txt.append(f"- GP: force_kernel_noise={gc['force_kernel_noise']}, prefer_UCB_if_sigma_ratio_low={gc['prefer_UCB_if_sigma_ratio_low']}")
        if plan["base_changes"]["force_acq"]:
            txt.append(f"- Force acquisition: {plan['base_changes']['force_acq']}")
        for n in plan["notes"]:
            txt.append(f"- Note: {n}")

        with open(os.path.join(fn_dir, "diagnostics.txt"), "w", encoding="utf-8") as f:
            f.write("\n".join(txt))
        print(f"[F{fn}-D{D}] 🩺 Diagnostics & plan saved → {fn_dir}")

        # Add to global plan
        global_plan[str(fn)] = plan

    # Global artifacts
    df = pd.DataFrame(diagnostics_rows)
    diag_csv = os.path.join(results_root, "global_diagnostics.csv")
    df.to_csv(diag_csv, index=False)
    with open(os.path.join(results_root, "global_rerun_plan.json"), "w") as f:
        json.dump(make_json_safe(global_plan), f, indent=2)

    print("\n📋 Global diagnostics CSV:", diag_csv)
    print("🧭 Global re-run plan JSON:", os.path.join(results_root, "global_rerun_plan.json"))

# ================================================================
# 1️⃣1️⃣  Automatic Re-run Wrapper (applies plans & reoptimizes)
# ================================================================
def auto_rerun_selected_functions(results_root="results_wk4_agentic_autonomous_plots"):
    """
    Reads global_rerun_plan.json and re-runs only those functions
    flagged as 'low_confidence' or 'unstable'. Applies parameter
    boosts (candidates, beta, w_ei, etc.) from the plan.
    """
    plan_path = os.path.join(results_root, "global_rerun_plan.json")
    if not os.path.exists(plan_path):
        print("\n⚠️ No global_rerun_plan.json found. Skipping auto re-run.")
        return

    with open(plan_path, "r") as f:
        plans = json.load(f)

    print("\n🚀 Starting Auto Re-run for selected functions...\n")
    dim_map = {1:2, 2:2, 3:3, 4:4, 5:4, 6:5, 7:6, 8:8}
    all_results = []

    for fn, plan in plans.items():
        fn = int(fn)
        D = dim_map[fn]
        prefix = f"[F{fn}-D{D}]"
        health_path = os.path.join(results_root, f"function_{fn}", "diagnostics.json")
        if not os.path.exists(health_path):
            continue
        health_data = json.load(open(health_path))
        health = health_data["health"]

        if health == "good":
            print(f"{prefix} ✅ Skipping (healthy run).")
            continue

        print(f"\n{prefix} ♻️ Re-running (Health={health}) with tuned parameters...")
        base_changes = plan["base_changes"]
        n_factor = base_changes.get("n_candidates_factor", 1.0)
        beta, w_ei, w_ucb, sigma_boost = (
            base_changes["beta"], base_changes["w_ei"],
            base_changes["w_ucb"], base_changes["sigma_boost"]
        )

        X0, y0, scale_factor = load_data_for_function(fn)
        cfg = AgentConfig(bounds=[(0, 1)] * D, n_candidates=int(3000 * n_factor))
        out_dir = os.path.join(results_root, f"function_{fn}_rerun")
        agent = HybridAgenticBO(X0, y0, cfg, out_dir, fn, D, scale_factor)
        agent.select_model(); agent.fit()

        MAX_ROUNDS = 5
        round_records, pi_list = [], []
        for r in range(1, MAX_ROUNDS + 1):
            mu_all, sigma_all = agent.predict_mu_sigma(agent.X)
            sigma_ratio = agent.choose_acquisition(mu_all, sigma_all)
            prop_x, prop_mu, prop_sig, prop_pi, prop_ei = agent.propose()
            reasoning = (
                f"{prefix} Round {r} (Re-run): Model={agent.model_name}, "
                f"Acq={agent.acq}, Mode={agent.explore_ctrl.mode}, "
                f"μ={prop_mu:.3e}, σ={prop_sig:.3e}, EI={prop_ei:.3e}, PI={prop_pi:.3f}"
            )
            print(f"{prefix} 🧠 {reasoning}")
            round_records.append({
                "round": r, "x": prop_x.tolist(), "mu": prop_mu,
                "sigma": prop_sig, "pi": prop_pi, "ei": prop_ei, "reasoning": reasoning
            })
            pi_list.append(prop_pi)
            if prop_pi > 0.7:
                print(f"{prefix} 🎯 High-confidence found (PI={prop_pi:.3f}) — stopping early.")
                break

        summary = {
            "function": fn, "dimension": D, "rounds_run": len(round_records),
            "avg_pi": np.mean(pi_list) if pi_list else 0, "records": round_records
        }
        with open(os.path.join(out_dir, "rerun_summary.json"), "w") as f:
            json.dump(make_json_safe(summary), f, indent=2)
        all_results.append(summary)

    print("\n✅ Auto re-run complete for flagged functions.\n")
    print("📦 Rerun summaries saved in *_rerun folders.")

# ================================================================
# 1️⃣2️⃣  Final Recommendations Report
# ================================================================
def generate_final_recommendations(results_root="results_wk4_agentic_autonomous_plots"):
    """
    Combines all summaries (main + re-run) to produce the final
    recommended query point and reasoning per function.
    """
    print("\n🧭 Generating Final Recommendations...\n")
    recs = []

    def load_json_if_exists(path):
        return json.load(open(path)) if os.path.exists(path) else None

    for fn in range(1, 9):
        fn_dir = os.path.join(results_root, f"function_{fn}")
        rerun_dir = os.path.join(results_root, f"function_{fn}_rerun")
        data_main = load_json_if_exists(os.path.join(fn_dir, "multi_round_summary.json"))
        data_rerun = load_json_if_exists(os.path.join(rerun_dir, "rerun_summary.json"))

        if data_rerun:
            last = data_rerun["records"][-1]
            used = "rerun"
        elif data_main:
            last = data_main["round_records"][-1]
            used = "main"
        else:
            continue

        fn_num = int(fn)
        D = (data_main or data_rerun)["dimension"]
        mu, sig, ei, pi = last["mu" if used == "rerun" else "predicted_mu"], \
                          last["sigma" if used == "rerun" else "predicted_sigma"], \
                          last["ei" if used == "rerun" else "predicted_ei"], \
                          last["pi" if used == "rerun" else "predicted_pi"]
        x_best = np.round(np.array(last["x" if used == "rerun" else "proposed_x"]), 6)
        reasoning = last.get("reasoning", "")
        recs.append({
            "function": fn_num, "dimension": D, "used_summary": used,
            "recommended_x": x_best.tolist(), "mu": mu, "sigma": sig,
            "ei": ei, "pi": pi, "reasoning": reasoning
        })
        print(f"[F{fn}-D{D}] ⭐ Final Recommendation ({used.upper()}):")
        print(f"  x*={x_best.tolist()}, μ={mu:.3e}, σ={sig:.3e}, EI={ei:.3e}, PI={pi:.3f}")
        print(f"  Reasoning: {reasoning}\n")

    df = pd.DataFrame(recs)
    csv_path = os.path.join(results_root, "final_recommendations.csv")
    df.to_csv(csv_path, index=False)
    print("\n📘 Final recommendations saved to:", csv_path)

# ================================================================
#  🔚 Main Execution
# ================================================================
if __name__ == "__main__":
    run_all_functions()                     # 1️⃣ initial autonomous optimization
    generate_agentic_summaries()            # 2️⃣ LLM-style summaries
    diagnose_runs()                         # 3️⃣ diagnostics + rerun plan
    auto_rerun_selected_functions()          # 4️⃣ execute rerun for flagged functions
    generate_final_recommendations()         # 5️⃣ produce final recommendations


[F1-D2] 🚀 Starting Function 1
[F1-D2] 📈 Previous best input: [0.731024 0.733   ]
[F1-D2] 📈 Previous best output: 7.710875e-16
[F1-D2] 🧠 Using Gaussian Process (best RMSE kernel)
[F1-D2] 🔧 Fitting GP ...
[F1-D2] ✅ Model training complete.

[F1-D2] 🔁 Round 1/8
[F1-D2] 🎯 Acquisition: UCB (σ_ratio=0.000)
[F1-D2] 💡 Proposed x=[0.793768 0.785155] μ=-1.420e-04, σ=9.383e-04, EI=4.933e-04, PI=0.440 (Mode=normal)
[F1-D2] 🧠 Reasoning: [F1-D2] Round 1. Model=GP, Acq=UCB, Mode=normal, μ=-1.420e-04, σ=9.383e-04, EI=4.933e-04, PI=0.440
[F1-D2] 🔧 Fitting GP ...
[F1-D2] ✅ Model training complete.

[F1-D2] 🔁 Round 2/8
[F1-D2] 🎯 Acquisition: UCB (σ_ratio=0.000)
[F1-D2] 💡 Proposed x=[0.432954 0.109905] μ=-2.458e-04, σ=9.329e-04, EI=3.769e-04, PI=0.330 (Mode=normal)
[F1-D2] 🧠 Reasoning: [F1-D2] Round 2. Model=GP, Acq=UCB, Mode=normal, μ=-2.458e-04, σ=9.329e-04, EI=3.769e-04, PI=0.330
[F1-D2] 🔧 Fitting GP ...
[F1-D2] ✅ Model training complete.

[F1-D2] 🔁 Round 3/8
[F1-D2] 🎯 Acquisition: UCB (σ_ratio=0.000)